[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/03_pretrained_korean/03_pretrained_korean.ipynb)

# 03. 사전 학습 한국어 모델 (KLUE-RoBERTa 파인튜닝)

## 이 장을 배우는 이유

02번은 **졌습니다.**

| | 검증 정확도 |
|---|---|
| TF-IDF + 로지스틱 회귀 (01번) | **0.8455** |
| 신경망 · 글자 단위 `Conv1D` (02번) | 0.7928 |

딥러닝을 썼는데 단어 세는 방식보다 5%p 낮았습니다. 02번은 그 이유까지 밝혀놨습니다.
**임베딩을 16,000건으로 처음부터 학습하기에는 데이터가 모자랐고, 검증 데이터 토큰의
38.5%가 사전에 없는 단어(`[UNK]`)로 뭉개졌습니다.**

그래서 02번은 이렇게 끝났습니다. *"그럼 언제부터 신경망이 나은가? 데이터가 많을 때,
그리고 **사전 학습 모델**을 쓸 때다."*

이번 장이 그 문장을 확인하는 자리입니다. **한국어를 이미 배운 모델을 가져와서,
우리 문제만 가르칩니다.**

이번 장에서 배우는 것

- **전이 학습**이 무엇이고 왜 데이터가 적을 때 이기는지
- 서브워드 토크나이저가 **OOV 문제를 어떻게 없애는지** (실측으로 확인합니다)
- `transformers`로 사전 학습 모델을 불러와 **파인튜닝**하는 법
- 그 대가는 무엇인지 — 크기, 시간, 그리고 **여전히 남는 한계**

## 이 노트북을 읽는 법

- **01번과 02번을 먼저 보세요.** 이 노트북은 그 두 결과와 **같은 분할에서 나란히 비교**하는 것이
  전부입니다. 비교 대상이 없으면 숫자가 그냥 숫자입니다.
- 셀을 위에서부터 순서대로 실행하세요(`Shift + Enter`).
- 실행 결과는 저장되어 있지 않습니다. 직접 실행해야 출력이 나타납니다.
- 코드 셀 앞에는 **지금 무엇을 할 것인지**, 뒤에는 **결과를 어떻게 읽는지**가 적혀 있습니다.

**소요 시간**: 읽는 데 40~50분. 학습 셀 하나가 오래 걸립니다.

> **GPU를 켜세요.** Colab 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`.
> 이 노트북의 숫자는 전부 **CPU에서 잰 값**이고, 학습 셀 하나에 **52분** 걸렸습니다.
> GPU에서는 이보다 훨씬 짧지만 얼마나 짧을지는 배정받는 런타임에 따라 다릅니다.
> 학습 셀이 직접 걸린 시간을 찍어주니 확인해보세요.

## 0. 준비

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q transformers pandas scikit-learn matplotlib koreanize-matplotlib

YNAT = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/ynat-v1.1"

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# GPU가 있으면 GPU로 보냅니다. 이 한 줄에 따라 학습 시간이 크게 갈립니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cpu":
    print("  (CPU입니다. 학습 셀이 오래 걸립니다. 런타임 유형을 T4 GPU로 바꾸는 걸 권합니다)")

## 1. 데이터 — 01·02번과 **완전히 같은 분할**

이번 장에서 가장 중요한 코드는 모델이 아니라 이 셀입니다.

01번 7절에서 배운 것을 떠올려보세요. **분할을 바꾸면 정확도가 0.01씩 흔들립니다.**
그러니 "사전 학습 모델이 0.8455를 이겼다"고 말하려면, 그 0.8455를 만든 것과
**똑같은 데이터, 똑같은 seed**로 학습해야 합니다. 그래야 [짝지은 비교](https://github.com/karzit/temp/blob/master/glossary.md#paired-comparison)가 됩니다.

아래 세 줄은 01번·02번의 것을 글자 하나 안 바꾸고 옮겨온 것입니다.

In [ ]:
raw = pd.read_json(f"{YNAT}/ynat-v1.1_train.json")
data = raw[["title", "label"]].sample(20_000, random_state=RANDOM_STATE).reset_index(drop=True)

X = data["title"].values
y_text = data["label"].values

X_train, X_valid, y_train_text, y_valid_text = train_test_split(
    X, y_text, test_size=0.2, stratify=y_text, random_state=RANDOM_STATE
)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)
y_valid = label_encoder.transform(y_valid_text)
N_CLASSES = len(label_encoder.classes_)

print("학습", len(X_train), "· 검증", len(X_valid), "·", N_CLASSES, "개 주제")
print("주제:", list(label_encoder.classes_))

**결과 읽는 법** — 학습 16,000 · 검증 4,000이 나와야 합니다. 01번 6절, 02번 1절과 같은 숫자입니다.
다르게 나온다면 `RANDOM_STATE`나 `sample` 크기를 건드린 것이니, 되돌리고 다시 실행하세요.
**여기가 어긋나면 이 노트북의 비교가 전부 무의미해집니다.**

## 2. 전이 학습 — 신입에게 무엇부터 가르치는가

02번 모델은 이런 신입이었습니다. **한국어를 한 글자도 모르는 채로 입사해서,
뉴스 제목 16,000건만 보고 "한국어"와 "주제 분류"를 **동시에** 배워야 했습니다.**

당연히 안 됩니다. 16,000건으로는 `정부는`과 `정부가`가 같은 말이라는 것조차 배울 수 없습니다.
02번에서 서로 다른 단어가 44,901개 나왔고 그중 74%가 딱 한 번만 등장했다고 했죠.
**한 번 본 단어의 뜻은 배울 수 없습니다.**

반면 이런 신입도 있습니다. **한국어는 이미 아는 사람.** 그에게 가르칠 것은 하나뿐입니다.
"이 제목이 경제인지 스포츠인지 골라라."

이것이 [전이 학습](https://github.com/karzit/temp/blob/master/glossary.md#transfer-learning)(transfer learning)입니다. 두 단계로 나뉩니다.

| 단계 | 누가 하나 | 무엇으로 | 무엇을 배우나 |
|---|---|---|---|
| **사전 학습**(pre-training) | 만든 사람 (KLUE 팀) | 위키·뉴스 등 **대량의 한국어** | 한국어 그 자체 — 어떤 단어가 어떤 자리에 오는지 |
| **파인튜닝**(fine-tuning) | 우리 | 우리 데이터 16,000건 | 이 문장이 7개 주제 중 무엇인지 |

우리가 하는 것은 두 번째 칸뿐입니다. 첫 칸은 GPU 수십 대로 며칠 걸리는 일이고,
**이미 끝나서 인터넷에 올라와 있습니다.**

이 노트북에서 쓸 모델은 **`klue/roberta-small`** 입니다. KLUE-YNAT를 만든 그 팀이
같이 공개한 한국어 모델입니다.

> **왜 하필 이 데이터에 이 모델인가.** KLUE-YNAT는 원래 **이런 모델들을 평가하려고**
> 만들어진 데이터셋입니다. 우리는 지금 그 벤치마크를 손으로 풀어보는 것입니다.
> `klue/roberta-base`(더 크고 더 정확함)로 바꾸는 것은 이름 한 줄만 고치면 됩니다.
> 여기서 `small`을 쓰는 이유는 **CPU에서도 끝나야 하기 때문**입니다.

## 3. 토크나이저 — 02번의 `[UNK]` 38.5%는 어디로 갔나

모델보다 먼저 볼 것이 있습니다. **글자를 어떻게 자르느냐**입니다.

02번은 공백으로 잘랐습니다. 그래서 `정부는`과 `정부가`가 **완전히 다른 단어**였고,
사전에 없는 단어는 전부 `[UNK]` 한 칸으로 뭉개졌습니다.

사전 학습 모델은 **서브워드**(subword)로 자릅니다. 단어를 통째로 담는 대신,
자주 나오는 **조각**을 담아두고 단어를 그 조각들로 조립합니다.
`정부는` → `정부` + `는` 처럼요.

말로만 들으면 안 와닿으니, 직접 잘라봅시다.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "klue/roberta-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("사전 크기:", f"{tokenizer.vocab_size:,}", "(02번의 단어 사전은 20,000이었습니다)\n")

for word in ["정부는", "정부가", "정부", "감시", "감시중"]:
    print(f"  {word:<5} → {tokenizer.tokenize(word)}")

**결과 읽는 법 — 01번 8절에서 문자 n-gram이 이겼던 이유가 여기 다시 나옵니다.**

`정부는`과 `정부가`가 **`정부`라는 조각을 공유**합니다. 02번의 단어 단위 사전에서는
둘이 남남이었지만, 여기서는 같은 뿌리를 나눠 갖습니다. 01번에서 문자 2~3-gram이
단어 단위를 8%p 앞섰던 것과 **같은 원리**입니다. 조사와 어미가 붙는 한국어에서
"단어를 통째로 사전에 넣는다"는 발상 자체가 손해였던 것입니다.

`감시`는 사전에 있는데 `감시중`은 없어서 통째로 `[UNK]`가 되던 02번의 문제도
여기서는 생기지 않습니다. 조각으로 조립하면 되니까요.

**그래서 OOV가 얼마나 줄었을까요?** 02번과 같은 자리(검증 데이터 4,000건)에서 재봅시다.

In [ ]:
from collections import Counter

# ① 02번 방식 — 학습 데이터에서 가장 많이 나온 단어 20,000개만 사전에 담고,
#    검증 데이터에서 그 사전에 없는 단어가 몇 %인지 센다.
사전 = {w for w, _ in Counter(w for s in X_train for w in s.lower().split()).most_common(20_000)}
검증단어 = [w for s in X_valid for w in s.lower().split()]
oov_word = sum(1 for w in 검증단어 if w not in 사전) / len(검증단어) * 100

# ② 03번 방식 — 서브워드로 자른 뒤 [UNK]가 몇 %인지 센다. 같은 검증 데이터입니다.
encoded = tokenizer(list(X_valid), add_special_tokens=False)["input_ids"]
flat = [i for row in encoded for i in row]
oov_sub = sum(1 for i in flat if i == tokenizer.unk_token_id) / len(flat) * 100

print("검증 데이터의 OOV 비율 (같은 데이터, 자르는 방식만 다름)")
print(f"  단어 단위 사전 20,000    {oov_word:5.2f}%")
print(f"  서브워드 사전 {tokenizer.vocab_size:,}   {oov_sub:5.2f}%")
print()
print(f"제목 하나당 토큰 수: 단어 {len(검증단어) / len(X_valid):.1f}개"
      f" → 서브워드 {len(flat) / len(X_valid):.1f}개")

**결과 읽는 법 — 여기가 이 장의 첫 번째 결정적 장면입니다.**

**38.8% → 0.16%.** 02번이 "근본 대책은 토큰 단위를 바꾸는 것"이라고 했던
그 말의 결과입니다. 검증 데이터의 거의 모든 글자가 이제 모델에게 **의미 있는 무언가**로 들어갑니다.

(02번은 이 값을 38.5%로 보고했습니다. 여기서 조금 다르게 나오는 것은 `TextVectorization`이
문장부호를 떼고 자르는 반면 위 코드는 공백으로만 잘랐기 때문입니다. **자르는 방식이 조금만
달라져도 숫자가 움직인다**는 것 자체가 이 장의 주제이기도 합니다.)

대신 제목 하나가 6.6개가 아니라 **13.4개 토큰**이 됐습니다. 단어를 조각내니 개수가 늘어난 것입니다.
계산량이 그만큼 늘어난다는 뜻이고, 이것이 나중에 볼 **대가**의 일부입니다.

**주의 — 아직 아무것도 이긴 게 아닙니다.** 지금 확인한 것은 "입력이 모델에게 제대로
전달된다"까지입니다. 02번 마지막의 경고를 그대로 옮기면, *입력이 모델에게 어떻게
보이는지 먼저 확인하라*는 단계를 통과한 것뿐입니다. 정확도는 아직 재지 않았습니다.

## 4. 모델 불러오기

`AutoModelForSequenceClassification`은 **사전 학습된 몸통 위에 분류용 머리를 얹어서**
돌려줍니다. 몸통은 한국어를 아는 상태로 오고, 머리는 비어 있는 상태로 새로 만들어집니다.
`num_labels=7`이 그 머리의 출력 칸 수입니다.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 {n_params:,}개")

**결과 읽는 법 — 빨간 경고가 나옵니다. 이건 정상입니다.**

> `Some weights of the model checkpoint were not used ...`
> `Some weights ... are newly initialized: ['classifier...']`

두 줄의 뜻은 이렇습니다.

- **not used**: 원래 모델에 달려 있던 "다음 단어 맞히기용 머리"(`lm_head`)를 떼어냈다
- **newly initialized**: 그 자리에 **주제 7개를 고르는 머리**를 새로 달았는데, 아직 아무것도 모른다

즉 **"몸통은 가져왔고 머리는 새것이니 이제 학습시켜라"**는 안내입니다.
이 경고가 안 나오면 오히려 이상합니다. 다만 진짜 문제일 때도 비슷한 문구가 나오므로,
**`classifier`가 아닌 층이 newly initialized로 뜬다면** 모델 이름을 잘못 적은 것입니다.

파라미터 개수도 보세요. 02번 모델들과 나란히 놓으면 이렇습니다.

| 모델 | 파라미터 |
|---|---|
| 02번 신경망 · 글자 단위 `conv` (02번의 최종 모델) | 225,415 |
| 02번 신경망 · 단어 단위 | 1,284,615 |
| **`klue/roberta-small`** | **68,096,263** |

02번의 최종 모델보다 **300배** 큽니다. 그런데 **그 대부분은 우리가 만든 것이 아닙니다.**
방금 새로 생긴 것은 맨 끝의 분류 머리뿐이고, 나머지 전부가 "한국어를 아는 부분"입니다.
2절의 표에서 첫 번째 칸(사전 학습)에 해당하는 것이 이 숫자입니다.

**크다는 것은 비용이기도 합니다.** 9절에서 이 크기가 무엇을 뜻하는지 다시 따집니다.

## 5. 학습 — 04번의 그 루프와 같은 모양

학습 코드는 `ml-curriculum` 04·05·06번에서 쓴 PyTorch 루프와 똑같습니다.
`zero_grad` → `forward` → `loss.backward()` → `optimizer.step()`.

달라지는 것은 두 가지뿐입니다.

- 입력이 이미지나 숫자가 아니라 **토큰 번호와 마스크** 두 개다
- 학습률이 아주 작다 — **3e-5**. 처음부터 배우는 게 아니라 **이미 아는 것을 조금만 조정**하는
  것이라, 크게 움직이면 사전 학습으로 얻은 것을 망가뜨립니다(catastrophic forgetting).

In [ ]:
MAX_LEN = 32   # 이 데이터에서 가장 긴 제목이 29토큰이라 32면 하나도 안 잘립니다 (해설 문제 2에서 확인)


def encode(texts):
    """문자열 목록 → (토큰 번호, 어텐션 마스크). 길이를 MAX_LEN으로 맞춥니다."""
    enc = tokenizer(list(texts), padding="max_length", truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]


train_ids, train_mask = encode(X_train)
train_labels = torch.tensor(y_train)

print("토큰 번호 모양:", tuple(train_ids.shape), " (문장 수, 길이)")
print("\n첫 문장:", X_train[0])
print("토큰    :", tokenizer.convert_ids_to_tokens(train_ids[0][:12]))
print("마스크  :", train_mask[0][:12].tolist(), " ← 1은 진짜 토큰, 0은 채워 넣은 자리")

**결과 읽는 법** — 토큰 목록 맨 앞의 `[CLS]`가 보이시나요? **문장 전체의 요약이 담길 자리**입니다.
분류 머리는 이 한 칸만 보고 주제를 고릅니다. 02번의 `GlobalAveragePooling1D`가 하던 일
(여러 벡터를 하나로 줄이기)을 여기서는 이 특별 토큰이 맡습니다.

마스크는 **어디까지가 진짜 문장인지** 알려줍니다. 32칸을 채우려고 뒤에 넣은 빈칸을
모델이 내용으로 착각하지 않게 하는 장치입니다.

이제 학습합니다. **오래 걸리는 셀입니다.**

In [ ]:
import time

BATCH, LR, EPOCHS = 32, 3e-5, 3
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)


@torch.no_grad()
def predict(texts, batch_size=128):
    """학습이 끝난 모델로 예측한다. 메모리 때문에 나눠서 넣습니다."""
    model.eval()
    ids, mask = encode(texts)
    out = []
    for i in range(0, len(ids), batch_size):
        logits = model(input_ids=ids[i:i + batch_size].to(device),
                       attention_mask=mask[i:i + batch_size].to(device)).logits
        out.append(logits.cpu())
    return torch.cat(out).argmax(dim=1).numpy()


started = time.time()
for epoch in range(EPOCHS):
    model.train()
    order = torch.randperm(len(train_ids))   # 매 epoch 순서를 섞는다
    total = 0.0

    for i in range(0, len(order), BATCH):
        batch = order[i:i + BATCH]
        optimizer.zero_grad()
        loss = model(input_ids=train_ids[batch].to(device),
                     attention_mask=train_mask[batch].to(device),
                     labels=train_labels[batch].to(device)).loss
        loss.backward()
        optimizer.step()
        total += loss.item() * len(batch)

    acc = (predict(X_valid) == y_valid).mean()
    print(f"epoch {epoch + 1}/{EPOCHS}  train_loss={total / len(order):.4f}"
          f"  valid_acc={acc:.4f}  ({time.time() - started:.0f}초)")

**결과 읽는 법**

- **1 epoch만에 이미 02번 신경망(0.7928)을 넘습니다.** 처음부터 배우지 않는다는 것의 차이입니다.
- **loss는 계속 떨어지는데 `valid_acc`는 그렇지 않습니다.** 이 노트북을 CPU에서 돌렸을 때는
  이렇게 나왔습니다.

| epoch | train_loss | valid_acc |
|---|---|---|
| 1 | 0.5245 | **0.8750** |
| 2 | 0.3057 | 0.8672 |
| 3 | 0.2169 | 0.8705 |

  **loss는 0.52 → 0.22로 절반 넘게 줄었는데 정확도는 오히려 1 epoch 때가 제일 높습니다.**
  이것이 [과적합](https://github.com/karzit/temp/blob/master/glossary.md#overfitting)의 교과서적인 모습입니다 — 모델이 학습 데이터를 외우기
  시작했고, 그 외운 것은 검증 데이터에 도움이 되지 않습니다. `ml-curriculum` 04번과
  `tabular-ml-practice` 04번에서 학습 곡선으로 본 그 장면이 여기서도 그대로 나옵니다.

- **그래서 이 데이터에서는 1~2 epoch면 충분합니다.** "3 epoch가 관례"라는 말을 그대로 따르기보다
  **직접 재보고 정하는 것**이 이 시리즈의 방식입니다. epoch마다 정확도를 찍어보는 이유가 이것입니다.
- 다만 0.8750과 0.8705의 차이는 01번 7절에서 본 **흔들림의 폭(0.01) 안**입니다.
  "1 epoch가 더 낫다"고 단정하려면 seed를 바꿔 여러 번 재봐야 합니다.

**이럴 때는 이걸 의심하세요.**

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| 정확도가 0.4 언저리에서 멈춘다 | 학습률이 너무 큼 | `LR`을 3e-5 이하로. 2e-5, 1e-5 (해설 문제 3에서 실측) |
| `CUDA out of memory` | 배치가 큼 | `BATCH`를 16이나 8로 |
| 한 epoch에 20분 넘게 걸린다 | CPU로 도는 중 | 런타임 유형을 T4 GPU로 |
| 정확도가 매번 크게 다르다 | 분류 머리가 새것이라 초기값 영향 | seed를 바꿔 두세 번 돌려 평균 보기 |

## 6. 정면 비교 — 세 방법을 같은 자리에 놓기

01번과 02번의 숫자를 그대로 옆에 놓습니다. **같은 20,000건, 같은 seed 42 분할**입니다.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

pred_valid = predict(X_valid)

print("검증 데이터 4,000건 (같은 분할)")
print(f"  01번 TF-IDF + 로지스틱 회귀   0.8455")
print(f"  02번 신경망 (글자 단위 conv)  0.7928")
print(f"  03번 사전 학습 모델 파인튜닝   {accuracy_score(y_valid, pred_valid):.4f}")
print()
print(f"  macro f1  {f1_score(y_valid, pred_valid, average='macro'):.4f}")

**결과 읽는 법 — 02번의 결론이 뒤집히는 지점입니다.**

**0.8705.** TF-IDF(0.8455)보다 **+0.0250**, 02번 신경망(0.7928)보다 크게 앞섭니다.

02번이 남긴 질문은 "딥러닝이 왜 졌나"였고, 답은 "데이터가 모자라서"였습니다.
이번 장에서 **데이터를 늘리지 않고** 이겼다는 데 주목하세요. 학습에 쓴 것은 여전히 16,000건입니다.
**모자랐던 데이터를 다른 사람이 이미 채워둔 것을 가져왔을 뿐입니다.**

01번 7절의 기준을 다시 적용해봅시다. 분할을 바꾸면 정확도가 0.01씩 흔들린다고 했습니다.
**+0.0250는 그 흔들림보다 훨씬 큽니다.** 그래서 이번에는 "개선됐다"고 말해도 됩니다.
0.001이 올랐다면 같은 말을 할 수 없었습니다.

## 7. 진짜 시험 — 분포가 다른 데이터

01번 12절에서 가장 아팠던 장면을 기억하시나요. 테스트에서 0.8455를 받은 모델이
평가 데이터(`dev`)에서는 **0.7639**로 떨어졌습니다. 학습 데이터의 `사회`가 11%인데
평가 데이터에서는 41%였기 때문입니다. [분포 이동](https://github.com/karzit/temp/blob/master/glossary.md#distribution-shift)입니다.

**사전 학습 모델은 여기서도 나을까요?** 이유가 있다면 이렇습니다 —
TF-IDF는 이 16,000건에서 본 단어만 알지만, 사전 학습 모델은 **훨씬 넓은 한국어를 보고 왔습니다.**
처음 보는 표현에 덜 당황할 것 같습니다. 확인해봅시다.

In [ ]:
dev = pd.read_json(f"{YNAT}/ynat-v1.1_dev.json")[["title", "label"]]
y_dev = label_encoder.transform(dev["label"])
pred_dev = predict(dev["title"].values)

# 01번 값은 그 노트북에서 가져온 상수이고, 03번 값은 방금 학습한 이 모델에서 계산합니다.
valid_acc = accuracy_score(y_valid, pred_valid)
dev_acc = accuracy_score(y_dev, pred_dev)

print(f"평가 데이터 {len(dev):,}건 (분포가 다름)")
print()
print("                    검증(같은 분포)   평가(다른 분포)    낙폭")
print(f"  01번 TF-IDF          0.8455           0.7639        -0.0816")
print(f"  03번 사전 학습 모델    {valid_acc:.4f}           {dev_acc:.4f}        {dev_acc - valid_acc:+.4f}")
print()
print(f"  macro f1 {f1_score(y_dev, pred_dev, average='macro'):.4f}")

**결과 읽는 법 — 숫자 두 개를 따로 읽어야 합니다.**

**① 절대 성능은 올랐습니다.** 0.7639 → **0.8345** (+0.0706).
분포가 달라진 데이터에서도 사전 학습 모델이 더 잘합니다.

**② 낙폭 자체도 줄었습니다.** 검증에서 평가로 넘어갈 때
TF-IDF는 -0.0816, 사전 학습 모델은 -0.0360입니다. **절반 이하입니다.**

**그래도 낙폭이 사라지지는 않았습니다.** -0.0360는 여전히 01번 7절의 흔들림 폭(0.01)보다
서너 배 큽니다.

여기까지는 숫자를 읽은 것입니다. **여기서 "사전 학습 모델은 분포 이동에 강하다"고 결론 내리고
싶어지는데, 아직 그렇게 말하면 안 됩니다.** 왜 안 되는지가 8절입니다. 먼저 주제별로 봅시다.

주제별로 어디가 약한지 봅시다.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_dev, pred_dev,
                            target_names=label_encoder.classes_, zero_division=0))

**결과 읽는 법 — 두 열을 함께 봐야 합니다.**

`support`는 그 주제가 평가 데이터에 몇 건 있는지입니다. **`사회`가 3,701건으로 41%를 차지합니다**
(학습 데이터에서는 11%였습니다). 그러니 `사회`의 f1이 전체 정확도를 좌우합니다.

CPU에서 잰 결과는 이렇게 나왔습니다.

| 주제 | f1 | support |
|---|---|---|
| 스포츠 | **0.926** | 578 |
| 세계 | 0.870 | 835 |
| 생활문화 | 0.848 | 1,369 |
| 사회 | 0.833 | **3,701** |
| 경제 | 0.818 | 1,348 |
| IT과학 | 0.787 | 554 |
| 정치 | **0.782** | 722 |

**가장 약한 칸이 `사회`가 아닙니다.** 01번에서 `사회`를 "딱 떨어지지 않는 것이 모이는
칸"(잔여 범주)이라고 불렀는데, 사전 학습 모델은 그 칸을 0.833까지 끌어올렸습니다.
대신 `정치`와 `IT과학`이 더 낮습니다. **`정치`와 `사회`는 사람이 봐도 경계가 흐린 쌍**이고,
`IT과학`은 건수 자체가 적습니다.

**스포츠가 0.926으로 압도적인 것**도 같은 이유로 읽힙니다. 선수 이름과 경기 용어가
다른 주제에 거의 안 나와서, 단어만 봐도 갈립니다. 01번의 TF-IDF도 이 주제는 잘 맞혔습니다.
**어떤 주제가 쉬운지는 모델이 아니라 데이터가 정합니다.**

01번 12절이 분포 이동의 대응책으로 **카테고리별 약점을 미리 파악해두기**를 들었는데,
이 표가 바로 그것입니다. 배포한 뒤 `정치` 문의가 갑자기 늘어난다면 미리 경고를 해둔 셈입니다.

## 8. 낙폭이 줄어든 진짜 이유 — 결론을 내리기 전에

7절에서 낙폭이 -0.0816 → -0.0360로 줄었습니다.
여기서 **"사전 학습 모델은 분포 이동에 강하다"** 고 결론 내리기 쉽습니다.
그런데 같은 숫자를 설명하는 **훨씬 시시한 이유**가 하나 있습니다.

> 평가 데이터는 `사회`가 41%입니다. 그런데 01번에서 `사회`는 **TF-IDF가 제일 못하던 주제**였습니다.
> 그러니 TF-IDF의 -0.0816는 "강건하지 못해서"가 아니라
> **"하필 자기가 제일 못하는 주제의 비중이 네 배로 커져서"** 일 수 있습니다.
> 그렇다면 사전 학습 모델이 잘한 것은 `사회` 하나뿐이고, 강건성과는 상관이 없습니다.

**두 설명은 데이터로 가릴 수 있습니다.** 검증 데이터에서 잰 주제별 실력이 그대로라고 치고,
**주제 비중만 평가 데이터의 것으로 바꿔서** 정확도를 계산해보면 됩니다.
그 값이 실제 평가 정확도와 같다면 낙폭은 전부 비중 탓이고, 강건성 이야기는 꺼낼 근거가 없습니다.

```
비중만 바뀌었다면 나왔을 정확도 = Σ (주제별 검증 recall × 그 주제의 평가 데이터 비중)
```

recall은 **그 주제 문장 중 몇 %를 맞혔는가**입니다. 주제 안에서 재는 값이라
비중이 바뀌어도 영향을 받지 않습니다. 그래서 이렇게 분리해볼 수 있습니다.

In [ ]:
from sklearn.metrics import recall_score

labels = list(label_encoder.classes_)

# 주제별 recall — 그 주제 문장 중 몇 %를 맞혔나. 주제 안에서 재므로 비중과 무관합니다.
rec_valid = recall_score(y_valid, pred_valid, labels=range(N_CLASSES), average=None, zero_division=0)
rec_dev = recall_score(y_dev, pred_dev, labels=range(N_CLASSES), average=None, zero_division=0)

mix_valid = np.bincount(y_valid, minlength=N_CLASSES) / len(y_valid)
mix_dev = np.bincount(y_dev, minlength=N_CLASSES) / len(y_dev)

print(f"{'주제':<7}{'검증 비중':>10}{'평가 비중':>10}{'검증 recall':>13}{'평가 recall':>13}")
for i, name in enumerate(labels):
    print(f"{name:<7}{mix_valid[i]:>10.3f}{mix_dev[i]:>10.3f}"
          f"{rec_valid[i]:>13.3f}{rec_dev[i]:>13.3f}")

# 실력은 그대로이고 비중만 바뀌었다면 나왔을 정확도
expected = float((rec_valid * mix_dev).sum())

print()
print(f"실제 검증 정확도            {accuracy_score(y_valid, pred_valid):.4f}")
print(f"실제 평가 정확도            {accuracy_score(y_dev, pred_dev):.4f}")
print(f"비중만 바뀌었다면 나왔을 값   {expected:.4f}")
print(f"비중으로 설명되지 않는 몫    {accuracy_score(y_dev, pred_dev) - expected:+.4f}")

**결과 읽는 법 — 예상과 반대 방향으로 어긋납니다.**

CPU에서 잰 값입니다. 01번 TF-IDF도 같은 계산을 해서 나란히 놓았습니다.

| | 실제 검증 | 실제 평가 | 비중만 바뀌었다면 | 설명 안 되는 몫 |
|---|---|---|---|---|
| 01번 TF-IDF | 0.8455 | 0.7639 | 0.7384 | +0.0255 |
| 03번 사전 학습 모델 | 0.8705 | 0.8345 | 0.7469 | **+0.0876** |

**비중 변화만 보면 낙폭이 실제보다 더 컸어야 합니다.** 두 모델 다 예상보다 잘했고,
사전 학습 모델이 그 이득을 세 배 넘게 가져갔습니다. 왜 그런지는 recall 표에서 드러납니다.

| 주제 | TF-IDF 검증 → 평가 | 사전 학습 검증 → 평가 |
|---|---|---|
| **사회** | 0.548 → **0.673** | 0.528 → **0.769** |
| IT과학 | 0.847 → 0.765 | 0.930 → 0.897 |
| 정치 | 0.918 → 0.845 | 0.933 → 0.888 |
| 스포츠 | 0.945 → 0.917 | 0.973 → 0.964 |

**다른 주제는 전부 내려가는데 `사회`만 크게 오릅니다.** 두 모델 모두요.
평가 데이터의 `사회`는 학습·검증 데이터의 `사회`와 **비중만 다른 것이 아니라 성격이 다릅니다.**
01번 12절이 "학습 11% → 평가 41%"를 지목했는데, 실제로는 그것만이 아니었던 것입니다.

**그리고 앞의 시시한 설명은 반증됩니다.** "사전 학습 모델이 `사회`를 잘한다"면
검증 데이터에서도 잘해야 하는데, **검증에서는 오히려 더 못합니다**(0.528 vs 0.548).
잘하는 것은 **평가 데이터의 `사회`** 입니다.

그래서 정확한 표현은 이렇게 됩니다.

> ❌ 사전 학습 모델은 분포 이동에 강하다
> ❌ 사전 학습 모델은 `사회`를 잘한다
> ⭕ 사전 학습 모델은 **처음 보는 성격의 문장을 훨씬 잘 따라간다**(+0.241 vs +0.125)

세 문장의 차이가 사소해 보이면 다시 읽어보세요. **첫 문장만 알고 있으면,
`사회`가 아니라 다른 주제의 성격이 바뀌었을 때 무슨 일이 생길지 예측할 수 없습니다.**

**이 절이 이 시리즈의 요약입니다.** 7절에서 정확도 두 개(0.8455 vs 0.8345)만 보고
"강건하다"고 쓸 뻔했습니다. 주제별로 쪼개보니 이야기가 달라졌습니다.
01번 7절에서 "0.001은 아무 의미 없다"고 배운 것과 같은 종류의 조심입니다 —
**요약된 숫자 하나는 그 안에서 무슨 일이 있었는지 말해주지 않습니다.**

## 9. 그래서 무엇을 얻고 무엇을 잃었나

이 시리즈를 관통하는 질문으로 돌아옵니다. **"성능이 올랐다고 말해도 되는가."**
이번에는 말해도 됩니다. 그런데 **공짜는 아닙니다.**

| | 01번 TF-IDF | 03번 사전 학습 모델 |
|---|---|---|
| 검증 정확도 | 0.8455 | **0.8705** |
| 평가 정확도 | 0.7639 | **0.8345** |
| 학습 시간 (CPU) | 1분 안쪽 | **52분** |
| 모델 크기 | 수~수십 MB | **약 270MB** (파라미터 6,800만 개 × 4바이트) |
| 예측 한 건 | 거의 즉시 | GPU 없으면 느림 |
| 왜 그렇게 답했는지 | **단어별 계수를 직접 볼 수 있음**(01번 6절) | 사실상 불가능 |
| 인터넷 없이 | 됨 | 모델을 미리 받아둬야 함 |

**두 열 중 무엇을 고를지는 정확도만으로 정해지지 않습니다.**

- 초당 수천 건을 분류해야 한다면 0.0250를 위해 수백 배 느린 모델을 쓸 수 없습니다
- "왜 이렇게 분류됐냐"는 질문에 답해야 하는 자리라면 01번 쪽이 낫습니다
- 라벨이 몇 백 건뿐이라면 — 그때는 사전 학습 모델이 **압도적입니다**. 다음 절에서 봅니다

그리고 **기준선을 먼저 만든 이유**가 여기서 완성됩니다. 01번을 건너뛰고 바로 이 노트북에
왔다면 0.8705가 좋은 숫자인지 알 수 없었을 것입니다. 비교 대상이 없으면 숫자는 그냥 숫자입니다.

## 10. 데이터를 줄이면 — 전이 학습이 진짜로 빛나는 자리

지금까지는 16,000건을 다 썼습니다. 그런데 현실에서 라벨이 붙은 데이터 16,000건은
**만들려면 사람이 16,000번 판단해야 하는 양**입니다. 보통은 그렇게 없습니다.

**데이터를 줄이면 어떻게 될까요?** 학습 데이터 앞에서부터 잘라서 다시 학습해봅니다.
비교 대상은 그대로 **16,000건으로 학습한 TF-IDF(0.8455)** 입니다.

In [ ]:
# 앞서 잰 결과입니다. 직접 돌려보려면 학습 셀을 n_train만 바꿔 반복 실행하세요.
# (모델을 매번 새로 불러와야 합니다 — 이어서 학습하면 앞의 데이터를 이미 본 상태가 됩니다)
결과 = { 1000: 0.8492, 2000: 0.8465, 4000: 0.8540, 16000: 0.8705 }

print("학습 건수   정확도    TF-IDF(16,000건 학습, 0.8455) 대비")
for n, acc in 결과.items():
    print(f"  {n:>6,}   {acc:.4f}   {acc - 0.8455:+.4f}")

**결과 읽는 법 — 여기서도 01번 7절의 기준을 그대로 적용해야 합니다.**

| 학습 건수 | 정확도 | TF-IDF(16,000건, 0.8455) 대비 |
|---|---|---|
| 1,000건 | 0.8492 | +0.0037 |
| 2,000건 | 0.8465 | +0.0010 |
| 4,000건 | 0.8540 | +0.0085 |
| 16,000건 | **0.8705** | +0.0250 |

**1,000건, 즉 TF-IDF가 쓴 양의 16분의 1로 0.8492가 나옵니다.**
그런데 이것을 "**넘었다**"고 말하면 안 됩니다. 차이가 +0.0037인데,
01번 7절에서 **분할만 바꿔도 정확도가 0.01씩 흔들린다**고 했습니다. 그 폭 안입니다.

정확한 표현은 **"1,000건으로 TF-IDF의 16,000건과 대등해진다"** 입니다.
1,000·2,000·4,000건 사이에서 순서가 뒤집히는 것(0.8492 → 0.8465 → 0.8540)도 같은 이유입니다.
**단조 증가가 아니라 흔들리는 것이고, 그 셋의 차이로는 아무 말도 할 수 없습니다.**

반면 16,000건(0.8705)과 1,000건(0.8492)의 차이 +0.0213는
흔들림 폭보다 큽니다. **"데이터가 더 있으면 더 낫다"는 말할 수 있고,
"2,000건보다 4,000건이 낫다"는 말할 수 없습니다.** 같은 표에서 어떤 비교는 되고
어떤 비교는 안 되는 것입니다.

**그래도 이 표가 전이 학습을 쓰는 가장 현실적인 이유를 보여줍니다.**
바뀌는 것은 정확도 0.0250가 아니라 **라벨을 몇 건 만들어야 하는가**입니다.
02번 방식으로 1,000건만 가지고 학습했다면 어땠을지 생각해보세요 —
16,000건으로도 0.7928이었습니다. 사람이 판단해야 하는 건수가 열 배 넘게 줄어드는 것은
정확도 몇 %p보다 훨씬 큰 차이입니다.

> **1,000~4,000건은 이 노트북과 별개로 잰 값입니다.** 같은 코드·같은 CPU에서 `n_train`만
> 바꿔 돌렸습니다. 그때 16,000건은 0.8708이 나왔고 이 노트북에서는 0.8705입니다 —
> **같은 코드를 두 번 돌린 차이가 0.0003**입니다. 이 표의 1,000·2,000·4,000건 차이를
> 함부로 읽으면 안 되는 이유가 하나 더 있는 셈입니다.

> **주의 — 이 실험은 앞에서부터 잘랐습니다.** 무작위 분할이라 주제 비율은 대체로 유지되지만,
> 1,000건이면 가장 작은 주제가 100건 남짓입니다. 진짜로 적은 데이터를 다룬다면
> `stratify`로 주제 비율을 맞춰 뽑고, seed를 바꿔 여러 번 재보세요.

## 정리

이번 장에서 한 일

1. 02번이 진 이유(OOV 38.8%)가 **서브워드 토크나이저에서 0.16%로 무너지는 것**을 확인했습니다
2. **전이 학습** — 사전 학습과 파인튜닝이 각각 무엇을 배우는지 구분했습니다
3. 01·02번과 **같은 분할**에서 파인튜닝해 0.8705를 얻었습니다 (TF-IDF 대비 +0.0250)
4. 분포가 다른 데이터에서 **절대 성능도 오르고 낙폭도 절반으로 줄었지만,
   낙폭이 사라지지는 않는다**는 것을 봤습니다
5. 그 낙폭을 **주제별로 분해해서**, "강건하다"는 결론이 성급했다는 것을 확인했습니다 —
   실제로 일어난 일은 **처음 보는 성격의 문장을 잘 따라간 것**이었습니다
6. 정확도 말고 **속도·크기·설명 가능성**까지 놓고 무엇을 고를지 따졌습니다

**스스로 확인해보기**

- [ ] 사전 학습과 파인튜닝이 각각 무엇을 배우는 단계인지 말할 수 있다
- [ ] 서브워드 토크나이저가 한국어에서 왜 유리한지, 01번의 문자 n-gram과 무엇이 닮았는지 설명할 수 있다
- [ ] 학습률을 3e-5처럼 작게 두는 이유를 안다
- [ ] `newly initialized` 경고가 왜 정상인지, 언제는 문제인지 구별할 수 있다
- [ ] "더 좋은 모델을 쓰면 분포 이동이 해결된다"가 왜 **반만 맞는지** 설명할 수 있다
- [ ] 낙폭이 줄어든 것을 보고 "강건하다"고 결론 내리면 안 되는 이유를 말할 수 있다
- [ ] "비중만 바뀌었다면 나왔을 정확도"를 어떻게 계산하는지, 그 값이 왜 필요한지 안다
- [ ] 정확도가 더 높은데도 01번 모델을 고를 수 있는 상황을 하나 댈 수 있다

## 연습 문제

1. **`klue/roberta-base`로 바꿔보세요.** `MODEL_NAME` 한 줄만 고치면 됩니다.
   정확도가 얼마나 오르고 학습 시간이 얼마나 늘어나는지 재보세요.
   **오른 정확도가 늘어난 시간만큼의 값어치가 있습니까?**
2. **`MAX_LEN`을 32에서 16으로 줄여보세요.** 제목 대부분이 그 안에 들어가는지
   3절의 토큰 수 통계로 먼저 확인하고, 정확도와 학습 시간이 어떻게 달라지는지 보세요.
3. **학습률을 3e-4로 열 배 키워보세요.** 무슨 일이 일어나는지 관찰하고,
   `ml-curriculum` 01번에서 배운 "학습률이 너무 클 때"와 같은 현상인지 비교해보세요.
4. **01번의 오분류 분석(11절)을 이 모델로 다시 해보세요.** `predict`의 결과로
   혼동 행렬을 그리고, 01번에서 TF-IDF가 헷갈리던 주제 쌍을 이 모델도 헷갈리는지 확인하세요.
   **같은 곳에서 틀린다면 그것은 모델의 한계가 아니라 데이터의 성질입니다.**
5. **앙상블을 시도해보세요.** 02번 6절에서 TF-IDF와 신경망을 섞었더니 오히려 내려갔습니다
   (실력 차이 때문이었습니다). 이번에는 두 모델의 실력이 더 가까운데, 섞으면 오를까요?
6. **자신의 텍스트 데이터로 이 노트북을 그대로 돌려보세요.** 고객 문의, 리뷰, 로그 메시지 —
   `X_train`/`y_train` 자리에 넣기만 하면 됩니다. 라벨이 500건밖에 없다면 9절의 결과를
   떠올려보세요.

**해설/정답**: [03_pretrained_korean_solutions.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/03_pretrained_korean/03_pretrained_korean_solutions.ipynb)

## 다음으로

**이 시리즈는 여기서 끝납니다.** 01번에서 기준선을 세우고, 02번에서 딥러닝이 지는 것을 보고,
03번에서 그 이유를 해결했습니다. 세 노트북을 관통한 질문은 하나였습니다 —
**"성능이 올랐다고 말해도 되는가."**

더 가볼 곳

- **형태소 분석기**(`kiwipiepy`)로 토큰화한 뒤 01번 8절의 문자 n-gram과 비교 —
  서브워드가 등장하기 전에 한국어를 자르던 방식입니다
- **`klue/roberta-large`** 또는 KLUE 논문에 실린 점수와 비교 — 우리가 푼 것이 그 벤치마크입니다
- **제목 대신 본문으로** — 01번 11절에서 "제목만으로는 알 수 없는 기사"가 오답의 한 축이었습니다
- [`rag-pipeline-practice`](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/rag-pipeline-practice/04_rag_pipeline/04_rag_pipeline.ipynb) —
  같은 "문장을 벡터로 바꾸기"를 **분류가 아니라 검색**에 쓰는 쪽입니다